# 05. Feature Extraction → ML-Ready CSV

This notebook converts all downloaded `.tif` files into a clean, flat CSV table ready for Machine Learning.

## Output Schema
| Point_ID | Latitude | Longitude | Elevation_m | Slope_deg | NDVI | LandCover | Rainfall_mm | Dist_River_m | Flood |
|---|---|---|---|---|---|---|---|---|---|

## How it Works
1. Sample **2000 random GPS points** across the Chikwawa boundary
2. For each of the **9 events**, drill each point through every `.tif` layer and record the pixel values
3. Calculate **distance to nearest river** using OSM river data
4. Stack all events → one big CSV with `9 events × 2000 points = ~18,000 rows`

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import osmnx as ox
from shapely.geometry import Point
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

# Paths
notebook_dir  = Path(os.path.abspath(''))
project_root  = notebook_dir.parent
raw_dir       = project_root / 'data' / 'raw'
events_dir    = raw_dir / 'events'
static_dir    = raw_dir / 'static'
processed_dir = project_root / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

print('Libraries loaded. Paths configured.')
print(f'  Events    : {events_dir}')
print(f'  Static    : {static_dir}')
print(f'  Output CSV: {processed_dir}')


Libraries loaded. Paths configured.
  Events    : c:\Users\linga\Music\Chikwawa-Flood-Prediction\data\raw\events
  Static    : c:\Users\linga\Music\Chikwawa-Flood-Prediction\data\raw\static
  Output CSV: c:\Users\linga\Music\Chikwawa-Flood-Prediction\data\processed


In [2]:
# STEP 1: Load Chikwawa boundary and sample 2000 random points inside it
RANDOM_SEED   = 42
NUM_POINTS    = 2000

boundary_path = raw_dir / 'chikwawa_boundary.geojson'
chikwawa_gdf  = gpd.read_file(boundary_path).to_crs('EPSG:4326')
district_geom = chikwawa_gdf.unary_union

print(f'Sampling {NUM_POINTS} random points inside Chikwawa...')
np.random.seed(RANDOM_SEED)
minx, miny, maxx, maxy = district_geom.bounds
sample_points = []
while len(sample_points) < NUM_POINTS:
    lon = np.random.uniform(minx, maxx)
    lat = np.random.uniform(miny, maxy)
    pt  = Point(lon, lat)
    if district_geom.contains(pt):
        sample_points.append({'Longitude': lon, 'Latitude': lat})

points_gdf = gpd.GeoDataFrame(
    sample_points,
    geometry=[Point(p['Longitude'], p['Latitude']) for p in sample_points],
    crs='EPSG:4326'
)
print(f'Done! {len(points_gdf)} points sampled.')


Sampling 2000 random points inside Chikwawa...
Done! 2000 points sampled.


In [3]:
# STEP 2: Extract static features (Elevation, Slope, LandCover)
# These do not change between events, so we extract them once.

def sample_raster_at_points(tif_path, points_gdf, band=1):
    """Sample pixel values from a GeoTIFF at each point location."""
    coords = [(row.geometry.x, row.geometry.y) for _, row in points_gdf.iterrows()]
    with rasterio.open(tif_path) as src:
        values = [x[0] if x[0] is not None else np.nan
                  for x in src.sample(coords, indexes=band)]
    return values

# Topography (Elevation Band 1, Slope Band 2)
topo_path = raw_dir / 'chikwawa_topography.tif'
print('Extracting Elevation and Slope...')
points_gdf['Elevation_m'] = sample_raster_at_points(topo_path, points_gdf, band=1)
points_gdf['Slope_deg']   = sample_raster_at_points(topo_path, points_gdf, band=2)
print(f'  Elevation range: {points_gdf["Elevation_m"].min():.1f} - {points_gdf["Elevation_m"].max():.1f} m')

# LandCover
lc_path = static_dir / 'landcover_esa2021.tif'
if lc_path.exists():
    print('Extracting LandCover...')
    points_gdf['LandCover'] = sample_raster_at_points(lc_path, points_gdf, band=1)
    print(f'  Unique LandCover classes: {sorted(points_gdf["LandCover"].dropna().unique().astype(int).tolist())}')
else:
    print('LandCover TIF not found - run notebook 04C first!')
    points_gdf['LandCover'] = np.nan

print('Static features extracted.')


Extracting Elevation and Slope...
  Elevation range: 48.0 - 955.0 m
Extracting LandCover...
  Unique LandCover classes: [10, 20, 30, 40, 50, 60, 80, 90]
Static features extracted.


In [ ]:
# STEP 3: Calculate Distance to Nearest River (Dist_River_m)
# Uses geopandas sjoin_nearest - vectorized with spatial index (fast!)
print('Fetching river/waterway network from OSM using boundary polygon...')

try:
    rivers_gdf = ox.features_from_polygon(district_geom, tags={'waterway': True})
    print(f'  Found {len(rivers_gdf)} waterway features.')

    # Keep only line/multiline geometries (rivers are lines, not points)
    rivers_gdf = rivers_gdf[rivers_gdf.geometry.geom_type.isin(
        ['LineString', 'MultiLineString']
    )].copy()
    print(f'  {len(rivers_gdf)} line features after filtering.')

    # Project both to UTM Zone 36S (metres) for accurate distance
    rivers_proj = rivers_gdf.to_crs('EPSG:32736')[['geometry']].reset_index(drop=True)
    points_proj  = points_gdf.to_crs('EPSG:32736')[['geometry']].copy()

    # sjoin_nearest is fully vectorized - uses spatial index, runs in seconds
    print('  Running vectorized nearest-river join (should take <30 seconds)...')
    joined = gpd.sjoin_nearest(
        points_proj,
        rivers_proj,
        how='left',
        distance_col='Dist_River_m'
    )
    # sjoin_nearest may duplicate index if multiple matches; take first
    joined = joined[~joined.index.duplicated(keep='first')]
    points_gdf['Dist_River_m'] = joined['Dist_River_m'].values

    print(f'  Distance range: {points_gdf["Dist_River_m"].min():.0f} - '
          f'{points_gdf["Dist_River_m"].max():.0f} m')
    print('  SUCCESS!')

except Exception as ex:
    print(f'OSM fetch failed: {ex}')
    print('Using Shire River approximation (known coordinates)...')
    from shapely.geometry import LineString
    import pyproj
    from shapely.ops import transform

    # Shire River main channel approximate centreline coordinates
    shire_coords = [
        (34.55, -15.80), (34.60, -15.90), (34.65, -16.00),
        (34.70, -16.10), (34.75, -16.20), (34.80, -16.30),
        (34.85, -16.40), (34.90, -16.50), (34.95, -16.60),
    ]
    shire_line = LineString(shire_coords)

    # Project to UTM 36S for metre-based distance
    proj = pyproj.Transformer.from_crs('EPSG:4326', 'EPSG:32736', always_xy=True).transform
    shire_proj = transform(proj, shire_line)
    points_proj = points_gdf.to_crs('EPSG:32736')
    points_gdf['Dist_River_m'] = points_proj.geometry.distance(shire_proj)
    print(f'  Approximation done. Range: {points_gdf["Dist_River_m"].min():.0f} '
          f'- {points_gdf["Dist_River_m"].max():.0f} m')

print('Dist_River_m column ready.')


Fetching river/waterway network from OSM using boundary polygon...
OSM fetch failed: name 'ox' is not defined
Using Shire River approximation (known coordinates)...


NameError: name 'points_gdf' is not defined

In [ ]:
# STEP 4: Define all 9 events and extract per-event features (Rainfall, NDVI, Flood)
EVENTS = [
    {'name': 'cyclone_bansi_jan2015',  'label': 'Cyclone Bansi 2015',  'is_flood': True},
    {'name': 'floods_feb2017',         'label': 'Minor Floods 2017',   'is_flood': True},
    {'name': 'cyclone_idai_mar2019',   'label': 'Cyclone Idai 2019',   'is_flood': True},
    {'name': 'cyclone_ana_jan2022',    'label': 'Cyclone Ana 2022',    'is_flood': True},
    {'name': 'cyclone_freddy_mar2023', 'label': 'Cyclone Freddy 2023', 'is_flood': True},
    {'name': 'normal_season_2016',     'label': 'Normal Season 2016',  'is_flood': False},
    {'name': 'normal_season_2018',     'label': 'Normal Season 2018',  'is_flood': False},
    {'name': 'normal_season_2020',     'label': 'Normal Season 2020',  'is_flood': False},
    {'name': 'normal_season_2021',     'label': 'Normal Season 2021',  'is_flood': False},
]

all_rows = []
point_id_counter = 1

for event in EVENTS:
    name      = event['name']
    event_dir = events_dir / name
    print(f'\nProcessing event: {event["label"]}')

    # Copy static columns into this event's snapshot
    event_df = points_gdf[['Latitude', 'Longitude', 'Elevation_m',
                            'Slope_deg', 'LandCover', 'Dist_River_m']].copy()
    event_df['Event'] = event['label']

    # Rainfall
    rain_path = event_dir / 'rainfall.tif'
    if rain_path.exists():
        event_df['Rainfall_mm'] = sample_raster_at_points(rain_path, points_gdf, band=1)
        print(f'  Rainfall extracted. Mean: {event_df["Rainfall_mm"].mean():.1f} mm')
    else:
        print(f'  WARNING: rainfall.tif not found for {name}')
        event_df['Rainfall_mm'] = np.nan

    # NDVI
    ndvi_path = event_dir / 'ndvi.tif'
    if ndvi_path.exists():
        event_df['NDVI'] = sample_raster_at_points(ndvi_path, points_gdf, band=1)
        print(f'  NDVI extracted. Mean: {event_df["NDVI"].mean():.3f}')
    else:
        print(f'  WARNING: ndvi.tif not found for {name}. Run updated 04C first!')
        event_df['NDVI'] = np.nan

    # Flood Mask (0 or 1)
    flood_path = event_dir / 'flood_mask.tif'
    if flood_path.exists():
        event_df['Flood'] = sample_raster_at_points(flood_path, points_gdf, band=1)
        flood_pct = (event_df['Flood'] == 1).mean() * 100
        print(f'  Flood mask extracted. {flood_pct:.1f}% of points are flooded.')
    else:
        print(f'  WARNING: flood_mask.tif not found for {name}')
        event_df['Flood'] = int(event['is_flood'])  # fallback: whole district

    # Assign Point_IDs
    event_df.insert(0, 'Point_ID',
        [f'P{str(i).zfill(5)}' for i in range(point_id_counter, point_id_counter + len(event_df))])
    point_id_counter += len(event_df)

    all_rows.append(event_df)

print(f'\nAll events processed.')


In [ ]:
# STEP 5: Combine all events into one CSV and save
final_df = pd.concat(all_rows, ignore_index=True)

# Reorder columns to match target schema
final_df = final_df[['Point_ID', 'Latitude', 'Longitude', 'Elevation_m',
                      'Slope_deg', 'NDVI', 'LandCover', 'Rainfall_mm',
                      'Dist_River_m', 'Flood', 'Event']]

# Cast types
final_df['Flood']     = final_df['Flood'].astype('Int64')  
final_df['LandCover'] = final_df['LandCover'].astype('Int64')

# Save to processed/
output_csv = processed_dir / 'chikwawa_flood_dataset.csv'
final_df.to_csv(output_csv, index=False)

print('='*55)
print('DATASET SAVED!')
print('='*55)
print(f'  File     : {output_csv}')
print(f'  Rows     : {len(final_df):,}')
print(f'  Columns  : {list(final_df.columns)}')
print(f'  Flooded  : {(final_df["Flood"]==1).sum():,} rows ({(final_df["Flood"]==1).mean()*100:.1f}%)')
print(f'  No Flood : {(final_df["Flood"]==0).sum():,} rows')
print()
print('Preview:')
final_df.head(10)
